# TSARA on the campaign archive — events on real data

**Phase 6, the companion to notebook 06.** Notebook 06 finds events in
air manufactured inside it, where every plume is known. This notebook runs the
same stage on the campaign archive, where nothing is known in advance, so you
can see it on the records it was built for:

1. **the ARC's Aeris**: one day of methane and ethane, on a car that drives and
   parks;
2. **a second analyzer of the same methane** on that car, and a column that
   turns out to be half copies;
3. **logs that never stop**: the ARC over nineteen days and the Wyoming van's
   Aeris over nine, where records are what keep one description of clean air
   from covering a week;
4. **one drive of the NOAA mobile lab**: methane and a VOC, a canister taking
   its events from a dense analyzer, and the drive numbers `METHODS.md` quotes.

Without an answer key, the evidence is of three kinds:

* **✔ checks against a loop written from the definition**, as in notebook 06.
  They hold on *any* day and at *any* parameters, so change them freely;
* **the numbers `docs/METHODS.md` §6.8 and §2.5 quote about the archive**,
  re-measured here from the files and collected in a **ledger** (section 5). A
  number is only *compared* when your parameters match the ones METHODS
  measured under; otherwise the ledger shows it and says so;
* **physical plausibility**, in the figures.

---

## How to use this notebook

1. **Point it at the archive.** Set `TSARA_ARCHIVE` to the directory holding the
   archive's `2024/` and `2026/` trees, then start Jupyter from that shell:

   ```bash
   export TSARA_ARCHIVE=/path/to/Data
   ```

2. **Run everything once** (*Run All*, a few minutes; the long logs of section 3
   and the 60 min window of section 1 are the slow parts).
3. **Each section loads what it reads**, in its own loading cell, whose
   parameters (such as `ARC_DAY`) need a re-run of that cell and the ones below
   it. Every other parameters cell needs only its own section re-run.
4. **"Try it"** notes suggest changes and say what should happen on the archive.

It reads only `2024/NOAA_ARC/`, `2024/NOAA_MobileLab_Drives/`,
`2024/NOAA_MobileLab_Stationary/` and `2026/03_instrument_aligned/`, and writes
only to a temporary directory. **It is committed without outputs**, because its
outputs are the archive's data.

---
## Setup

The toolkit: the archive gate, imports, the figure style shared by notebooks
01–06, the reference implementations the ✔ checks compare TSARA against, and the
ledger that compares re-measured numbers with `METHODS.md`.

In [ ]:
# =============================================================================
# TOOLKIT. Every section needs this cell.
# =============================================================================
import logging
import os
import tempfile
from fractions import Fraction
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tsara import setup_logging
from tsara.baseline import baseline_state, rolling_quantile, window_cells
from tsara.config.analysis import BaselineConfig, EventsConfig
from tsara.config.loader import load_manifest
from tsara.core.support import CellBounds, stream_cells
from tsara.events import event_catalog, event_state, event_states, find_events, find_records
from tsara.ingest import ingest_campaign

# --- the archive, and only the parts of it this notebook may read -------------
ARCHIVE_ENV = "TSARA_ARCHIVE"
if not os.environ.get(ARCHIVE_ENV):
    raise RuntimeError(
        f"This notebook reads the campaign archive, and {ARCHIVE_ENV} is not set. Set it to "
        "the directory holding the archive's 2024/ and 2026/ trees, e.g. "
        f"`export {ARCHIVE_ENV}=/path/to/Data`, start Jupyter from that shell, and run again. "
        "Notebook 06 runs the same stage on generated data and needs nothing."
    )
ARCHIVE = Path(os.environ[ARCHIVE_ENV])
ARC = ARCHIVE / "2024" / "NOAA_ARC"
DRIVES = ARCHIVE / "2024" / "NOAA_MobileLab_Drives"
GROUND = ARCHIVE / "2024" / "NOAA_MobileLab_Stationary"
ALIGNED = ARCHIVE / "2026" / "03_instrument_aligned"
for tree in (ARC, DRIVES, GROUND, ALIGNED):
    if not tree.is_dir():
        raise RuntimeError(
            f"{ARCHIVE_ENV} is set, but it holds no {tree.relative_to(ARCHIVE)}. "
            "Point it at the directory containing 2024/ and 2026/."
        )

_work = tempfile.TemporaryDirectory()
WORK = Path(_work.name)
SECOND = 1_000_000_000  # TSARA keeps every time as integer nanoseconds
HOUR = 3600 * SECOND

# --- figure style (identical to notebooks 01-06) -------------------------------
C1, C2, C3 = "#0072b2", "#e69f00", "#009e73"  # Okabe-Ito blue, amber, green: colour-blind safe
C4 = "#cc79a7"  # Okabe-Ito purple, for a fourth series
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, SURFACE = "#e1e0d9", "#fcfcfb"
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.linewidth": 0.8,
        "axes.labelcolor": INK2,
        "axes.titlecolor": INK,
        "axes.titlesize": 11,
        "axes.titleweight": "normal",
        "axes.titlelocation": "left",
        "axes.labelsize": 9.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "grid.color": GRID,
        "grid.linewidth": 0.7,
        "grid.linestyle": "-",
        "legend.frameon": False,
        "legend.fontsize": 9,
        "font.size": 9.5,
        "lines.linewidth": 1.4,
        "figure.dpi": 110,
    }
)


def finish(
    ax, title=None, sub=None, ylab=None, xlab=None, legend=False, grid_axis="y", loc="upper right"
):
    """Apply the shared chrome: left-aligned title, muted subtitle, hairline grid."""
    if title:
        ax.set_title(title, pad=19 if sub else 8)
    if sub:
        ax.text(0, 1.025, sub, transform=ax.transAxes, fontsize=8.8, color=MUTED, va="bottom")
    if ylab:
        ax.set_ylabel(ylab)
    if xlab:
        ax.set_xlabel(xlab)
    if grid_axis:
        ax.grid(True, axis=grid_axis, alpha=0.9)
    ax.set_axisbelow(True)
    if legend:
        ax.legend(loc=loc)


def hhmm(ax, fmt="%H:%M"):
    """Label the x axis as clock time; the date belongs in the title."""
    ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))


def break_gaps(clock, values, tolerance=5.0):
    """Insert a NaN wherever the clock jumps, so matplotlib draws no line across a gap."""
    clock, values = np.asarray(clock), np.asarray(values, dtype=float)
    steps = np.diff(clock).astype("timedelta64[ms]").astype(float)
    gaps = np.where(steps > tolerance * np.median(steps))[0]
    if gaps.size == 0:
        return clock, values
    return np.insert(clock, gaps + 1, clock[gaps]), np.insert(values, gaps + 1, np.nan)


def shade(ax, rows, color=C2, alpha=0.3):
    """Shade each catalog row's interval, start to end."""
    for start, end in zip(rows["start_time"], rows["end_time"]):
        ax.axvspan(start, end, color=color, alpha=alpha, lw=0)


# --- loading -----------------------------------------------------------------
def manifest(name, body):
    """Write a manifest into the scratch directory and load it (validated by Pydantic)."""
    path = WORK / f"{name}.yaml"
    path.write_text(body)
    return load_manifest(path)


def finite_only(stream, variable):
    """Return the rows of a stream where `variable` holds a value, cells kept.

    The 2024 merge files put every instrument on 1 s rows with a blank
    wherever it had nothing to report; the analyzer's own readings are the
    rows that hold one. `isel` keeps the (time, nv) cells.
    """
    keep = np.flatnonzero(np.isfinite(stream[variable].values))
    return stream.isel(time=keep)


def sweep(windows=("2min", "10min", "60min"), quantiles=(0.05,), **more):
    """Return a BaselineConfig for a sweep."""
    return BaselineConfig.model_validate(
        {"windows": list(windows), "quantiles": list(quantiles), **more}
    )


def held_share(values):
    """Share of finite readings exactly equal to the previous finite reading."""
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.mean(x[1:] == x[:-1])) if x.size > 1 else float("nan")


def overlapping(rows, others):
    """Per row, whether its [start, end] interval overlaps any of `others`'."""
    a0, a1 = rows["start_time"].to_numpy()[:, None], rows["end_time"].to_numpy()[:, None]
    b0, b1 = others["start_time"].to_numpy()[None, :], others["end_time"].to_numpy()[None, :]
    return np.any((b0 <= a1) & (a0 <= b1), axis=1)


def seen_by(rows, stream, variable):
    """Per row, whether `variable` has a finite reading inside the row's interval."""
    ok = np.isfinite(stream[variable].values)
    clock = stream["time"].values[ok]
    first = np.searchsorted(clock, rows["start_time"].to_numpy())
    last = np.searchsorted(clock, rows["end_time"].to_numpy(), side="right")
    return last > first


# --- reference implementations (share no code with TSARA) --------------------
# Written from METHODS §6.8, deliberately slow and plain, as in notebook 06.


def reference_half_sample_mode(values):
    """Bickel and Fruhwirth (2006), one interval at a time (earliest on a tie)."""
    x = sorted(v for v in values if np.isfinite(v))
    while len(x) > 3:
        half = -(-len(x) // 2)
        best, width = 0, np.inf
        for j in range(len(x) - half + 1):
            if x[j + half - 1] - x[j] < width:
                best, width = j, x[j + half - 1] - x[j]
        x = x[best : best + half]
    if len(x) < 3:
        return float(np.mean(x))
    if x[1] - x[0] < x[2] - x[1]:
        return (x[0] + x[1]) / 2
    if x[1] - x[0] > x[2] - x[1]:
        return (x[1] + x[2]) / 2
    return x[1]


def reference_clean(values):
    """Return the clean level and spread of one record at one sweep point, by definition."""
    level = reference_half_sample_mode(values)
    below = [level - v for v in values if np.isfinite(v) and v < level]
    return level, 1.4826 * float(np.median(below))


def reference_records(midpoints_ns, finite, gap_ns, max_ns):
    """Find records one reading at a time, part edges as exact fractions (METHODS §6.8)."""
    kept = [i for i in range(len(midpoints_ns)) if finite[i]]
    pieces = []
    for i in kept:
        if pieces and int(midpoints_ns[i]) - int(midpoints_ns[pieces[-1][-1]]) <= gap_ns:
            pieces[-1].append(i)
        else:
            pieces.append([i])
    out, record = np.full(len(midpoints_ns), -1), 0
    for piece in pieces:
        first = int(midpoints_ns[piece[0]])
        span = int(midpoints_ns[piece[-1]]) - first
        n = max(1, -(-span // max_ns))  # the fewest parts no longer than the maximum
        while Fraction(span, n) > max_ns:
            n += 1
        parts = {}
        for i in piece:
            k = min(int(Fraction(int(midpoints_ns[i]) - first) * n / span) if span else 0, n - 1)
            parts.setdefault(k, len(parts))
            out[i] = record + parts[k]
        record += len(parts)
    return out


def reference_dropouts(midpoints_ns, record):
    """Per reading, whether the step back to the previous reading of its record is a dropout."""
    out = np.zeros(len(midpoints_ns), dtype=bool)
    steps_of = {}
    previous = {}
    for i, r in enumerate(record):
        if r < 0:
            continue
        if r in previous:
            steps_of.setdefault(r, []).append(
                (i, int(midpoints_ns[i]) - int(midpoints_ns[previous[r]]))
            )
        previous[r] = i
    for steps in steps_of.values():
        typical = np.median([step for _, step in steps])
        for i, step in steps:
            out[i] = step > 1.5 * typical
    return out


def reference_events(z, cells, dropout_before, record, enter, exit_, gap_ns):
    """Events one reading at a time, from METHODS §6.8's paragraph (as in notebook 06)."""
    membership = np.full(len(z), -1)
    events, current, dip, previous = [], [], [], None

    def close():
        if current and max(z[i] for i in current) >= enter:
            events.append(list(current))

    for i in range(len(z)):
        if record[i] < 0:
            continue
        broken = (
            previous is None
            or record[i] != record[previous]
            or dropout_before[i]
            or np.isnan(z[i])
            or np.isnan(z[previous])
        )
        previous = i
        if broken:
            close()
            current, dip = [], []
        if np.isnan(z[i]):
            continue
        if z[i] >= exit_:
            if current and dip and max(cells.start_ns[i] - cells.stop_ns[current[-1]], 0) >= gap_ns:
                close()
                current, dip = [], []
            current = current + dip + [i]
            dip = []
        elif current:
            dip.append(i)
    close()
    for number, rows in enumerate(events):
        membership[rows] = number
    return membership


# --- the noise estimate Phase 6 rejected, re-measured (METHODS §2.5) ----------
def diff_mad_estimate(stream, instrument, variable, window, factor=1.5, min_samples=10):
    """Return the robust first-difference noise estimate at every reading.

    The estimator of METHODS §2.5 (code at commit 59bd6ef), rejected as the
    events stage's scale and kept here only to re-measure what it would have
    done on a drive. A sample is the absolute difference of two consecutive
    finite readings, on the cell between their midpoints, dropped when they
    are farther apart than `factor` times the median spacing; the window
    centred on each reading takes the weighted median of the samples touching
    it (TSARA's rolling engine, the one §2.5 describes), scaled by
    1.4826 / sqrt(2), blank with fewer than `min_samples`, floored at
    delta / sqrt(12).
    """
    cells = stream_cells(stream, instrument)
    values = np.asarray(stream[variable].values, dtype=float)
    finite = np.flatnonzero(np.isfinite(values))
    middle = cells.midpoint_ns[finite]
    spacing = np.diff(middle)
    keep = spacing <= factor * np.median(spacing)
    samples = np.abs(np.diff(values[finite]))[keep]
    sample_cells = CellBounds(start_ns=middle[:-1][keep], stop_ns=middle[1:][keep])
    rolled = rolling_quantile(
        sample_cells, samples, window_cells(cells, pd.Timedelta(window).value), [0.5]
    )
    sigma = np.where(
        rolled.n_readings >= min_samples, 1.4826 * rolled.values[:, 0] / np.sqrt(2.0), np.nan
    )
    steps = np.diff(np.unique(values[finite]))
    floor = steps[steps > 0].min() / np.sqrt(12.0) if (steps > 0).any() else 0.0
    return np.where(np.isfinite(sigma), np.maximum(sigma, floor), np.nan)


# --- the ledger and the checks -----------------------------------------------
LEDGER = {}  # claim -> (section, documented, measured, status)
CHECKS = {}  # claim -> held?


def documented(section, claim, value, measured, fmt="{}", applies=True, conditions=""):
    """Put a number METHODS quotes beside the same number re-measured just now.

    Compared as printed, at the precision METHODS prints it. `applies` says
    whether the parameters match the ones METHODS measured under; when they do
    not, the number is shown and not compared.
    """
    shown_doc, shown_now = fmt.format(value), fmt.format(measured)
    status = ("agrees" if shown_doc == shown_now else "DIFFERS") if applies else "not compared"
    LEDGER[claim] = (section, shown_doc, shown_now, status)
    note = "" if applies else f"; METHODS measured {conditions}"
    print(f"  [{status:>12}] {claim}: {shown_now}  (METHODS {section}: {shown_doc}{note})")


def check(claim, holds, detail=""):
    """Record one claim about TSARA and print whether it held just now."""
    CHECKS[claim] = bool(holds)
    mark = "✔" if holds else "✘ DOES NOT HOLD:"
    print(f"  {mark} {claim}" + (f"   [{detail}]" if detail else ""))


# --- logging, with the archive and scratch paths hidden ----------------------
def hide_paths(record):
    """Show the archive and scratch directories as <archive> and <work> in TSARA's log messages."""
    for real, shown in ((str(WORK), "<work>"), (str(ARCHIVE), "<archive>")):
        if isinstance(record.msg, str):
            record.msg = record.msg.replace(real, shown)
        if isinstance(record.args, tuple):
            record.args = tuple(
                str(a).replace(real, shown) if real in str(a) else a for a in record.args
            )
    return True


for handler in setup_logging(logging.WARNING).handlers:
    handler.addFilter(hide_paths)

ARC_DAYS = sorted(p.name.split("_")[2] for p in ARC.glob("USOS-ARL-Suite_ARC_*.ict"))
print("archive found; ARC days available:", ", ".join(ARC_DAYS))

---
## 1. The ARC's Aeris: one day of methane and ethane

The ARC is NOAA's car, and its suite writes one file a day with a row every
second: an Aeris laser analyzer's methane and ethane beside two Picarros and
the car's position and speed. The car logs all day, parked most of it, so a day
is a long record with a few drives in it.

The events stage runs at its defaults (`EventsConfig()`: entry 3, exit 1, dips
under 5 s bridged, records split at gaps over 2 h and cut to 6 h), on a 5th
percentile baseline at three windows. What to look for: a clean spread per
record, which is the same air's wobble whether the car is parked or driving;
how it compares with the figure the file's header states for the analyzer; how
many events each window finds against the chance column; and whether methane's
events are ethane's.

One thing the file itself says must be done first. Every three hours the suite
switches the Aeris's inlet to a zero and then a span gas for about a minute and
a half, and marks it in a column: `Valve`, "0 is measurement; 10 is zeroing; 11
is spanning" by the file's header. A span gas is a plume to the events stage.
The manifest masks those readings with a `flag` rule on that column, the tool
QA/QC has for exactly this (`METHODS.md` §9.5). A `range` rule does the rest: on
some days the analyzer faults and reports methane near −3.7 ppm and ethane near
−50 ppb for hours (all of 07-25 and most of 07-26), which no air holds. Methane
below 1500 ppb and ethane below −10 ppb are masked, the ethane bound far below
the analyzer's noise around zero.

### Load one ARC day (sections 1 and 2 read it)

In [ ]:
# ---- PARAMETERS (loading) -----------------------------------------------------
ARC_DAY = "20240730"  # any of ARC_DAYS; 07-30 has the most driving
MASK_CALIBRATION = True  # mask the Aeris's zero and span periods (Valve not 0)

In [ ]:
# QA/QC, one line of YAML per Aeris variable: a physical range, and (when asked)
# the flag rule that keeps only rows whose Valve is 0 (10 zeroing, 11 spanning).
valve = ", {kind: flag, flag_column: Valve, good_values: [0]}" if MASK_CALIBRATION else ""
ch4_rules = f"        qaqc: [{{kind: range, min: 1500.0, max: 100000.0}}{valve}]\n"
c2h6_rules = f"        qaqc: [{{kind: range, min: -10.0}}{valve}]\n"
arc_day = ingest_campaign(
    manifest(
        "arc_day",
        f"""
name: arc_{ARC_DAY}
base_path: {ARC}
platform: {{kind: mobile, gps_instrument: arc}}
instruments:
  arc:
    loader: {{format: icartt, path_template: "USOS-ARL-Suite_ARC_{ARC_DAY}_*.ict"}}
    variables:
      latitude: {{column: lat_DGPS_deg, role: gps_lat, units: degrees_north}}
      longitude: {{column: lon_DGPS_deg, role: gps_lon, units: degrees_east}}
      speed: {{column: speed_km_h, role: aux, units: km/h}}
      valve: {{column: Valve, role: aux, units: "1"}}
      ch4_aeris:
        column: CH4_aeris313_ppm
        role: gas
        units: ppb
        field: ch4
        convert: {{from_unit: ppm, to_unit: ppb, scale: 1000.0}}
{ch4_rules}      c2h6_aeris:
        column: C2H6_aeris313_ppb
        role: gas
        units: ppb
        field: c2h6
{c2h6_rules}      ch4_g2401:
        column: CH4_g2401m_ppm
        role: gas
        units: ppb
        field: ch4
        convert: {{from_unit: ppm, to_unit: ppb, scale: 1000.0}}
""",
    )
)["arc"]
moving = np.nanmean(arc_day["speed"].values > 5)
print(
    f"{ARC_DAY}: {arc_day.sizes['time']} rows; Aeris methane in "
    f"{np.isfinite(arc_day['ch4_aeris'].values).sum()}, G2401 methane in "
    f"{np.isfinite(arc_day['ch4_g2401'].values).sum()}; moving faster than 5 km/h in {moving:.0%}"
)
calibrating = np.nan_to_num(arc_day["valve"].values) != 0
print(
    f"the valve calibrates in {calibrating.sum()} rows; masked in the Aeris's methane by rule: "
    f"{arc_day['ch4_aeris'].attrs.get('qaqc_masked')}, in its ethane: "
    f"{arc_day['c2h6_aeris'].attrs.get('qaqc_masked')}"
)
if calibrating.any():
    there = np.nanmedian(arc_day["ch4_g2401"].values[calibrating])
    print(
        f"the G2401 reads a median {there:.0f} ppb while the valve calibrates: "
        "ambient air, so the valve is the Aeris's alone"
    )
check(
    "§1 with the mask, no Aeris reading survives inside a calibration period",
    not MASK_CALIBRATION
    or not (
        np.isfinite(arc_day["ch4_aeris"].values[calibrating]).any()
        or np.isfinite(arc_day["c2h6_aeris"].values[calibrating]).any()
    ),
    f"{calibrating.sum()} rows",
)

In [ ]:
# ---- PARAMETERS (section 1) ---------------------------------------------------
WINDOWS = ("2min", "10min", "60min")
QUANTILE = 0.05
STATED = {"ch4_aeris": 1.0, "c2h6_aeris": 0.2}  # the file header's UNCERTAINTY, ppb

In [ ]:
aeris = ("ch4_aeris", "c2h6_aeris")
state = baseline_state(
    arc_day, instrument="arc", baseline=sweep(WINDOWS, (QUANTILE,)), variables=list(aeris)
)
found = event_state(state, instrument="arc", events=EventsConfig())
catalog = event_catalog({"arc": found}, {"arc": state})
seconds = [pd.Timedelta(w).total_seconds() for w in WINDOWS]

for v in aeris:
    record = found[f"record_{v}"].values
    n = int(np.isfinite(arc_day[v].values).sum())
    first = [int(np.flatnonzero(record == r)[0]) for r in range(int(record.max()) + 1)]
    print(f"\n{v}: {n} readings in {len(first)} records; the header states {STATED[v]:g} ppb")
    print(
        f"  {'window':>7}  {'clean spread per record (ppb)':>34}  "
        f"{'events':>7} {'chance':>7}  in events"
    )
    for w, window in enumerate(WINDOWS):
        spreads = " ".join(f"{found[f'clean_spread_{v}'].values[i, w, 0]:6.3f}" for i in first)
        events = int(found[f"n_events_{v}"].values[w, 0, 0, 0])
        chance = float(found[f"chance_events_{v}"].values[w, 0, 0, 0])
        share = float(np.sum(found[f"event_{v}"].values[:, w, 0, 0, 0] >= 0) / n)
        print(f"  {window:>7}  {spreads:>34}  {events:>7} {chance:>7.0f}  {share:.0%}")

# Methane's events and ethane's, at each window: do they overlap?
for w, window in enumerate(WINDOWS):
    at = catalog["baseline_window"] == seconds[w]
    ch4 = catalog[at & (catalog["species"] == "ch4_aeris")]
    c2h6 = catalog[at & (catalog["species"] == "c2h6_aeris")]
    print(
        f"at {window}: {overlapping(ch4, c2h6).mean():.0%} of methane's events overlap an "
        f"ethane event, {overlapping(c2h6, ch4).mean():.0%} of ethane's overlap a methane event"
    )

# Checks: the clean description and the events, from the definitions.
matches = []
for v in aeris:
    record = found[f"record_{v}"].values
    for w in range(len(WINDOWS)):
        delta = state[f"enhancement_{v}"].values[:, w, 0]
        for r in range(int(record.max()) + 1):
            rows = np.flatnonzero(record == r)
            level, spread = reference_clean(list(delta[rows]))
            matches.append(
                level == found[f"clean_level_{v}"].values[rows[0], w, 0]
                and spread == found[f"clean_spread_{v}"].values[rows[0], w, 0]
            )
check(
    "§1 every record's clean level and spread are the definition's, at every window",
    all(matches),
    f"{len(matches)} (variable, window, record)",
)
cells = stream_cells(arc_day, "arc")
middle = cells.midpoint_ns
for v in aeris:
    finite = np.isfinite(arc_day[v].values)
    by_hand = reference_records(middle, finite, 2 * HOUR, 6 * HOUR)
    check(
        f"§1 {v}'s records are the definition's, reading by reading",
        np.array_equal(found[f"record_{v}"].values, by_hand),
    )
    w = WINDOWS.index("10min") if "10min" in WINDOWS else 0
    loop = reference_events(
        found[f"z_{v}"].values[:, w, 0],
        cells,
        reference_dropouts(middle, by_hand),
        by_hand,
        3.0,
        1.0,
        5 * SECOND,
    )
    check(
        f"§1 {v}'s events at {WINDOWS[w]} are a loop over the definition's, reading by reading",
        np.array_equal(found[f"event_{v}"].values[:, w, 0, 0, 0], loop),
    )

In [ ]:
# Figure only: the day, methane and ethane with their events at 10 min, and the car's speed.
w = WINDOWS.index("10min") if "10min" in WINDOWS else 0
t = arc_day["time"].values
fig, axes = plt.subplots(
    3, 1, figsize=(10, 6.4), sharex=True, gridspec_kw={"height_ratios": [2, 2, 1], "hspace": 0.25}
)
for ax, v, label, colour in zip(axes, aeris, ("methane (ppb)", "ethane (ppb)"), (C1, C3)):
    rows = catalog[(catalog["species"] == v) & (catalog["baseline_window"] == seconds[w])]
    shade(ax, rows)
    clock, values = break_gaps(t, arc_day[v].values)
    ax.plot(clock, values, color=colour, lw=0.5)
    low, high = np.nanpercentile(arc_day[v].values, [0.5, 99.5])
    ax.set_ylim(low - 0.1 * (high - low), high + 0.3 * (high - low))
    finish(ax, None, None, label)
axes[0].set_title(
    f"The ARC on {ARC_DAY}: the Aeris's methane and ethane, and the events found at {WINDOWS[w]}",
    pad=19,
)
axes[0].text(
    0,
    1.025,
    "shaded: events; the 0.5th-99.5th percentiles fill each panel, so the tallest plumes run off",
    transform=axes[0].transAxes,
    fontsize=8.8,
    color=MUTED,
    va="bottom",
)
clock, values = break_gaps(t, arc_day["speed"].values)
axes[2].plot(clock, values, color=INK2, lw=0.6)
finish(axes[2], None, None, "speed (km/h)")
hhmm(axes[2])
plt.show()

**What to read off the output.** The clean spread is a few ppb of methane and a
few tenths of a ppb of ethane. At 2 min ethane's is close to the 0.2 ppb the
file's header states for the analyzer and methane's is about twice its 1 ppb;
both grow at longer windows, where the air's slow wobble joins the instrument's
noise (`METHODS.md` §6.8: the spread is not a measurement uncertainty), and
they differ from record to record because the air does. Every window finds many
times the chance column's count: the car sits in a city. Nine in ten of
ethane's events overlap a methane event, but methane's events cover most of the
day, so an overlap alone cannot say two gases came from one source; a ratio
fitted over the events can, and that is Phase 7's.

**Try it.**
- `MASK_CALIBRATION = False`: the span gas comes back as a spike every three
  hours, in both gases at once (the zero gas stays masked: it reads below
  1500 ppb, and the range rule catches it).
- `ARC_DAY = "20240726"`: the analyzer faulted for most of that day, and the
  range rule masks those hours, so a day of about 85,000 rows keeps about 30,000
  Aeris readings; the checks still hold on what is left.
- `WINDOWS = ("2min",)` runs in seconds.

---
## 2. A second analyzer of the same methane, and a column that is half copies

The ARC's suite also carries a Picarro G2401's methane. Two analyzers of one gas
on one car are the best test TSARA gets without an answer key: each variable
finds its own events on its own readings (`METHODS.md` §6.8), so where both see
a plume both should report it.

But look at the G2401's column first. In the merge file its value is exactly
the previous row's in about half the rows, to ten decimal places, which no
analyzer measures twice: the G2401 reports every 2 s or so, and the suite writes
its last value into each 1 s row until the next arrives. A held copy is not a
reading, and TSARA cannot tell one from a reading (`METHODS.md` §9.2.3). What it
does to this stage: tied readings put the baseline, a quantile, exactly on a
reading, so many enhancements are exactly zero and the clean level can land on
zero; and every held copy counts as a reading in the chance column, which
assumes independent readings.

In [ ]:
# ---- PARAMETERS (section 2) ---------------------------------------------------
WINDOW = "10min"  # the window at which the two analyzers' events are compared
QUANTILE = 0.05

In [ ]:
analyzers = ("ch4_aeris", "ch4_g2401")
for v in (*analyzers, "c2h6_aeris"):
    print(f"{v}: {held_share(arc_day[v].values):.1%} of readings equal the previous reading")

pair_windows = tuple(dict.fromkeys(("2min", WINDOW)))  # 2 min, where ties show, and WINDOW
pair_state = baseline_state(
    arc_day, instrument="arc", baseline=sweep(pair_windows, (QUANTILE,)), variables=list(analyzers)
)
pair = event_state(pair_state, instrument="arc", events=EventsConfig())
pair_catalog = event_catalog({"arc": pair}, {"arc": pair_state})
for v in analyzers:
    record = pair[f"record_{v}"].values
    first = [int(np.flatnonzero(record == r)[0]) for r in range(int(record.max()) + 1)]
    for w, window in enumerate(pair_windows):
        delta = pair_state[f"enhancement_{v}"].values[:, w, 0]
        zero = np.mean(delta[np.isfinite(delta)] == 0.0)
        levels = " ".join(f"{pair[f'clean_level_{v}'].values[i, w, 0]:6.2f}" for i in first)
        spreads = " ".join(f"{pair[f'clean_spread_{v}'].values[i, w, 0]:5.2f}" for i in first)
        print(
            f"  {v} at {window:>5}: enhancement exactly 0 at {zero:5.1%}; "
            f"clean level per record {levels}; spread {spreads}"
        )

# Compare only events the other analyzer could have seen: those holding one of its readings.
at = pair_catalog["baseline_window"] == pd.Timedelta(WINDOW).total_seconds()
a = pair_catalog[at & (pair_catalog["species"] == "ch4_aeris")]
g = pair_catalog[at & (pair_catalog["species"] == "ch4_g2401")]
a, g = a[seen_by(a, arc_day, "ch4_g2401")], g[seen_by(g, arc_day, "ch4_aeris")]
print(
    f"\nat {WINDOW}, events the other analyzer had readings in: Aeris {len(a)}, G2401 {len(g)}; "
    f"{overlapping(a, g).mean():.0%} of the Aeris's overlap a G2401 event, "
    f"{overlapping(g, a).mean():.0%} of the G2401's an Aeris event"
)
for label, rows, other in (("Aeris", a, g), ("G2401", g, a)):
    big = rows["peak_z"] > 10
    print(
        f"  of the {label}'s events standing more than 10 clean spreads high ({int(big.sum())}), "
        f"{overlapping(rows[big], other).mean():.0%} overlap an event of the other"
    )

**What to read off the output.** The Aeris's column holds a new value almost
every second; the G2401's repeats in about half its rows, and at 2 min its
enhancement is exactly zero many times as often as the Aeris's, which is the tie
at work: a clean level of exactly 0.00 is the ties, not the air. Compared only
where the other analyzer had readings, the two agree at 10 min on nearly every
event, and on every event standing more than 10 clean spreads high: two
instruments of one gas, detected separately, find the same plumes. They are not
equally quiet, so each is judged against its own clean spread, and at longer
windows the two descriptions of clean air part (the Try-it below).

Held copies are a property of the merge file, not of the analyzer, and TSARA
reads what the file says. A manifest cannot say "this column is held", and
whether TSARA should learn to drop held repeats is an open question, not
decided here. Until then, choose the analyzer that reports at the rows' rate,
as section 1 does.

**Try it.** `WINDOW = "60min"`: the Aeris's large events are still the G2401's,
but under half of the G2401's events, and a third of its large ones, overlap an
Aeris event. At an hour the G2401's clean spread in the first record is less
than half the Aeris's (the printed rows), and a smaller spread lets smaller
excursions through.

---
## 3. Logs that never stop: records

The clean level and spread describe plume-free air one **record** at a time: a
variable's readings split where two are more than `record_gap` apart (default
2 h) and cut into equal parts no longer than `max_record_length` (default 6 h,
about one drive; `METHODS.md` §6.8). The cut exists because the gap rule alone
does not split a record that logs continuously, and some of the archive does:
the ARC's suite logs parked and driving alike, and the Wyoming van's Aeris
logged for a week without a two-hour gap. One description of clean air over a
week would mix nights with afternoons.

Five streams, each one variable's finite readings in time order: the ARC's
Aeris over its nineteen days (with section 1's QA/QC), the Wyoming van's 2026
Aeris and Picarro, and the 2024 mobile lab's Picarro while driving (ten drive
days) and parked at its ground site (60 s means over 35 days). The last three
are `METHODS.md`'s table, and go to the ledger.

### Load the long logs (section 3 reads them; about half a minute)

In [ ]:
def one_stream(name, base, block):
    """Ingest one instrument and return its stream, finite readings of `ch4` only."""
    stream = ingest_campaign(
        manifest(
            name,
            f"name: {name}\nbase_path: {base}\n"
            "platform: {kind: stationary, latitude: 40.77, longitude: -111.89}\n"
            f"instruments:\n  log:\n{block}",
        )
    )["log"]
    return finite_only(stream, "ch4")


logs = {
    "ARC Aeris, 2024": one_stream(
        "arc",
        ARC,
        """
    loader: {format: icartt, path_template: "USOS-ARL-Suite_ARC_*.ict"}
    variables:
      ch4:
        column: CH4_aeris313_ppm
        role: gas
        units: ppb
        convert: {from_unit: ppm, to_unit: ppb, scale: 1000.0}
        qaqc:
          - {kind: range, min: 1500.0, max: 100000.0}
          - {kind: flag, flag_column: Valve, good_values: [0]}
""",
    ),
    "Wyoming van Aeris, 2026": one_stream(
        "wyo_aeris",
        ALIGNED,
        """
    loader:
      format: parquet
      path_template: "WYO_aerisultra460/Eng/*.parquet"
      exclude: ["**/bad/**", "**/bad_timestamp/**"]
    variables:
      ch4: {column: CH4_ppm, role: gas, units: ppm}
""",
    ),
    "Wyoming van Picarro, 2026": one_stream(
        "wyo_picarro",
        ALIGNED,
        """
    loader:
      format: parquet
      path_template: "WYO_picarro/*.parquet"
      exclude: ["**/bad/**", "**/bad_timestamp/**"]
    variables:
      ch4: {column: CH4_sync, role: gas, units: ppm}
""",
    ),
    "2024 drive Picarro, ten drives": one_stream(
        "drives",
        DRIVES,
        """
    loader: {format: icartt, path_template: "*/USOS-Picarro-CO2-CH4-CO-H2O_MobileLab_*.ict"}
    variables:
      ch4: {column: CH4_ppb, role: gas, units: ppb}
""",
    ),
    "2024 ground Picarro, 60 s means": one_stream(
        "ground",
        GROUND,
        """
    loader:
      format: icartt
      path_template: "*/USOS-Picarro-CO2-CH4-CO-H2O_MobileLabGround_*.ict"
      support: {stop_column: StopTime_UTC, method: mean}
    variables:
      ch4: {column: CH4_ppb_1min, role: gas, units: ppb}
""",
    ),
}
for name, stream in logs.items():
    print(f"{name}: {stream.sizes['time']:,} readings")

In [ ]:
# ---- PARAMETERS (section 3) ---------------------------------------------------
RECORD_GAP = "2h"
MAX_RECORD_LENGTH = "6h"

In [ ]:
gap_ns, max_ns = pd.Timedelta(RECORD_GAP).value, pd.Timedelta(MAX_RECORD_LENGTH).value
AT_DOCUMENTED_GAP = RECORD_GAP == "2h"
no_cap = 10**6 * HOUR  # the gap rule alone


def durations_h(records, middle):
    """Each record's span, first reading's midpoint to last's, in hours."""
    return np.array([np.ptp(middle[records.index == r]) / HOUR for r in range(records.n)])


print(f"{'':32} {'readings':>10} {'span':>7}   {'gap rule alone':>31}   {'and the cap':>28}")
table = {}
for name, stream in logs.items():
    cells = stream_cells(stream, "log")
    middle = cells.midpoint_ns
    finite = np.ones(middle.size, dtype=bool)
    alone = find_records(cells, finite, gap_ns=gap_ns, max_length_ns=no_cap)
    capped = find_records(cells, finite, gap_ns=gap_ns, max_length_ns=max_ns)
    d_alone, d_capped = durations_h(alone, middle), durations_h(capped, middle)
    span_d = np.ptp(middle) / (24 * HOUR)
    table[name] = (middle.size, span_d, alone.n, d_alone, cells)
    print(
        f"{name:32} {middle.size:>10,} {span_d:>5.1f} d   "
        f"{alone.n:>3} records, {d_alone.min():5.1f}/{np.median(d_alone):5.1f}/"
        f"{d_alone.max():6.1f} h   {capped.n:>3} records, {d_capped.min():4.1f}/"
        f"{np.median(d_capped):4.1f}/{d_capped.max():4.1f} h"
    )
print("(durations: min / median / max)")


def as_printed(d):
    """Return a duration as METHODS's table prints it: hours, or days from 100 hours up."""
    return f"{d / 24:.1f} d" if d >= 100 else f"{d:.1f} h"


documented_rows = {
    "2024 drive Picarro, ten drives": ("87,272 over 29.5 d", 10, "3.9 / 5.5 / 7.2 h"),
    "2024 ground Picarro, 60 s means": ("45,987 over 34.9 d", 11, "16.7 h / 41.8 h / 11.7 d"),
    "Wyoming van Picarro, 2026": ("331,588 over 9.3 d", 3, "5.9 h / 6.9 h / 7.2 d"),
}
for name, (readings, count, durations) in documented_rows.items():
    n, span_d, n_records, d, _ = table[name]
    documented("§6.8", f"{name}: readings, span", readings, f"{n:,} over {span_d:.1f} d")
    documented(
        "§6.8",
        f"{name}: records at gaps over 2 h",
        count,
        n_records,
        "{}",
        AT_DOCUMENTED_GAP,
        "at 2 h",
    )
    if name.startswith("2024 drive"):
        shown = f"{d.min():.1f} / {np.median(d):.1f} / {d.max():.1f} h"
    else:
        shown = " / ".join(as_printed(x) for x in (d.min(), np.median(d), d.max()))
    documented(
        "§6.8",
        f"{name}: durations min / median / max",
        durations,
        shown,
        "{}",
        AT_DOCUMENTED_GAP,
        "at 2 h",
    )
drive_cells = table["2024 drive Picarro, ten drives"][4]
counts = [
    find_records(
        drive_cells,
        np.ones(len(drive_cells), bool),
        gap_ns=pd.Timedelta(g).value,
        max_length_ns=no_cap,
    ).n
    for g in ("10min", "30min", "2h", "6h")
]
documented(
    "§6.8",
    "ten drives: records at gaps over 10 min / 30 min / 2 h / 6 h",
    "10 / 10 / 10 / 10",
    " / ".join(map(str, counts)),
)

# Check: the records of the week-long log are the definition's, reading by reading.
name = "Wyoming van Aeris, 2026"
cells = table[name][4]
records = find_records(cells, np.ones(len(cells), bool), gap_ns=gap_ns, max_length_ns=max_ns)
by_hand = reference_records(cells.midpoint_ns, np.ones(len(cells), bool), gap_ns, max_ns)
check(
    f"§3 the {name}'s records are the definition's, reading by reading",
    np.array_equal(records.index, by_hand),
    f"{len(cells):,} readings",
)

In [ ]:
# Figure only: the week-long log, its methane above and its records below.
name = "Wyoming van Aeris, 2026"
stream = logs[name]
cells = table[name][4]
records = find_records(cells, np.ones(len(cells), bool), gap_ns=gap_ns, max_length_ns=max_ns)
t = stream["time"].values
fig, (ax, bar) = plt.subplots(
    2, 1, figsize=(10, 4.6), sharex=True, gridspec_kw={"height_ratios": [2.4, 1], "hspace": 0.12}
)
clock, values = break_gaps(t, stream["ch4"].values, tolerance=600)
ax.plot(clock, values, color=C1, lw=0.4)
low, high = np.nanpercentile(stream["ch4"].values, [0.5, 99.5])
ax.set_ylim(low - 0.05 * (high - low), high + 0.2 * (high - low))
finish(
    ax,
    f"{name}: {len(cells):,} readings, split at gaps over {RECORD_GAP} "
    f"and cut to {MAX_RECORD_LENGTH}",
    f"{records.n} records; alternating colours; a white space is a gap longer than {RECORD_GAP}",
    "methane (ppm)",
)
for r in range(records.n):
    bar.axvspan(
        pd.Timestamp(int(records.start_ns[r])),
        pd.Timestamp(int(records.stop_ns[r])),
        ymin=0.15,
        ymax=0.85,
        color=(C1, C2)[r % 2],
        lw=0,
    )
bar.set_yticks([])
bar.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
finish(bar, None, None, "records", None, grid_axis="x")
plt.show()

**What to read off the output.** The gap rule alone finds the drives: ten
records for ten drive days, the same at any gap from 10 min to 6 h, because a
drive's analyzer is off between drives. It does not split what logs without
stopping: the ground site's Picarro runs eleven days without a two-hour gap,
the Wyoming van's analyzers a week, the ARC's Aeris four days. The cap cuts
each piece into equal parts no longer than six hours, so every description of
clean air covers at most a drive's worth of air, and the figure shows a week
cut into its parts. Cutting is a choice with a cost: an event running across a
cut is two events, one in each record.

**Try it.** `MAX_RECORD_LENGTH = "24h"`: each continuous piece is cut into days
instead: the week-long logs into records of just under a day, the ground site's
eleven days into twelve, and the drives, all shorter than a day, not at all.
`RECORD_GAP = "10min"`: the ground site splits at its shorter outages (27
records by the gap rule alone, not 11), while the drives do not change.

---
## 4. One drive of the NOAA mobile lab: methane, a VOC, and a canister

`METHODS.md` §6.8 chose detection's scale on this drive, the 2024-07-18 drive of
the NOAA mobile lab, and quotes what it measured there: how plume-free air's
level grows with the window, how much of the drive sits inside an event under
the scale chosen (C, the clean spread) and under the one rejected (B, a rolling
noise estimate, §2.5), how many events are a single reading, and why a reported
uncertainty column is not used as a floor. This section re-measures each under
METHODS's rule (5th percentile; entry 3, exit 1; nothing bridged; the drive one
record per analyzer), leading with the Picarro's methane and the PTR-MS's
benzene; the CO₂ and NOy numbers METHODS also quotes are measured the same way
and go straight to the ledger.

It ends with the canister. The iWAS fills a canister for 15 s every few
minutes, about thirty a drive: far too few to describe its own clean air, so it
takes its events from the PTR's benzene, named as its trigger.

### Load one drive

In [ ]:
# ---- PARAMETERS (loading) -----------------------------------------------------
DRIVE_DAY = "20240718"  # the drive METHODS measured; any of the ten drive days runs

In [ ]:
AT_DOCUMENTED_DAY = DRIVE_DAY == "20240718"  # whether ledger numbers can be compared
drive = ingest_campaign(
    manifest(
        "one_drive",
        f"""
name: drive_{DRIVE_DAY}
base_path: {DRIVES / DRIVE_DAY}
platform: {{kind: mobile, gps_instrument: metnav}}
instruments:
  picarro:
    loader: {{format: icartt, path_template: "USOS-Picarro-CO2-CH4-CO-H2O_MobileLab_*.ict"}}
    variables:
      ch4: {{column: CH4_ppb, role: gas, units: ppb}}
      co2: {{column: CO2_ppm, role: gas, units: ppm}}
  lif:
    loader: {{format: icartt, path_template: "USOS-NOy-LIF_MobileLab_*.ict"}}
    variables:
      noy:
        column: NOy_LIF
        role: gas
        units: ppb
        convert: {{from_unit: pptv, to_unit: ppb, scale: 0.001}}
        # By the file's own header a calibration figure, not a precision (METHODS 6.7).
        uncertainty:
          systematic: {{mode: reported, column: NOy_LIF_1SigmaAccuracy}}
  ptr:
    loader: {{format: icartt, path_template: "USOS-PTR_MobileLab_*.ict"}}
    variables:
      benzene_ptr: {{column: Benzene_NOAAPTR_ppbv, role: gas, units: ppb, field: benzene}}
  metnav:
    loader: {{format: icartt, path_template: "USOS-MetNav_MobileLab_*.ict"}}
    variables:
      latitude: {{column: GPS_Lat_deg, role: gps_lat, units: degrees_north}}
      longitude: {{column: GPS_Lon_deg, role: gps_lon, units: degrees_east}}
  iwas:
    loader:
      format: icartt
      path_template: "USOS-iWAS_MobileLab_*.ict"
      support: {{stop_column: iWAS_Stop_UTC, method: mean}}
    variables:
      benzene_iwas: {{column: Benzene_ppbv, role: gas, units: ppb, field: benzene}}
      toluene_iwas: {{column: Toluene_ppbv, role: gas, units: ppb, field: toluene}}
""",
    )
)
# Each analyzer's own readings: the merge rows where it reported, its sigma columns
# set aside (the clean spread does not read them; NOy's is used below on purpose).
variables = {"ch4": "picarro", "benzene_ptr": "ptr", "co2": "picarro", "noy": "lif"}
readings = {v: finite_only(drive[i], v) for v, i in variables.items()}
noy_reported = readings["noy"]["sigma_sys_noy"].values.astype(float)
for v in readings:
    readings[v] = readings[v].drop_vars(
        [n for n in readings[v].data_vars if str(n).startswith("sigma_")]
    )
print(
    f"{DRIVE_DAY}: "
    + "; ".join(f"{v} {s.sizes['time']:,} readings" for v, s in readings.items())
    + f"; iWAS {drive['iwas'].sizes['time']} fills"
)

In [ ]:
# ---- PARAMETERS (section 4) ---------------------------------------------------
WINDOWS = ("2min", "10min", "60min")
QUANTILE = 0.05
WINDOW = "10min"  # the window METHODS's in-event table is measured at

In [ ]:
AT_RULE = (
    AT_DOCUMENTED_DAY
    and WINDOWS == ("2min", "10min", "60min")
    and QUANTILE == 0.05
    and WINDOW == "10min"
)
rule = EventsConfig(max_internal_gap="1ns", max_record_length="24h")  # nothing bridged, one record
w = WINDOWS.index(WINDOW)
states, found, catalogs, share_c, share_b, local = {}, {}, {}, {}, {}, {}
for v, instrument in variables.items():
    s = readings[v]
    states[v] = baseline_state(
        s, instrument=instrument, baseline=sweep(WINDOWS, (QUANTILE,)), variables=[v]
    )
    found[v] = event_state(states[v], instrument=instrument, events=rule)
    catalogs[v] = event_catalog({instrument: found[v]}, {instrument: states[v]})
    n = s.sizes["time"]
    share_c[v] = float(np.sum(found[v][f"event_{v}"].values[:, w, 0, 0, 0] >= 0) / n)
    # Meaning B: a rolling noise estimate, the offset modelled as z_q sigma, the same detector.
    local[v] = diff_mad_estimate(s, instrument, v, WINDOW)
    delta = states[v][f"enhancement_{v}"].values[:, w, 0]
    z_b = (delta - 1.645 * local[v]) / local[v]
    cells = stream_cells(s, instrument)
    one = find_records(cells, np.ones(n, bool), gap_ns=24 * HOUR, max_length_ns=24 * HOUR)
    share_b[v] = float(
        np.sum(
            find_events(z_b, cells, one, enter=3.0, exit_=1.0, max_internal_gap_ns=0).membership
            >= 0
        )
        / n
    )

heading = "in events at " + WINDOW + ", C / B"
print(f"{'':12} {'clean level per window (ppb)':>30}   {heading:>24}   events   one reading")
for v in ("ch4", "benzene_ptr"):
    levels = " ".join(
        f"{found[v][f'clean_level_{v}'].values[0, k, 0]:8.3g}" for k in range(len(WINDOWS))
    )
    rows = catalogs[v][catalogs[v]["baseline_window"] == pd.Timedelta(WINDOW).total_seconds()]
    print(
        f"{v:12} {levels:>30}   {share_c[v]:>11.0%} / {share_b[v]:<9.0%}   {len(rows):>6}   "
        f"{(rows['n_readings'] == 1).mean():.1%}"
    )

# The ledger: METHODS §6.8's drive numbers, then §2.5's.
for k, window in enumerate(("2min", "10min", "60min")):
    if window in WINDOWS:
        documented(
            "§6.8",
            f"the drive's methane's clean level at {window} (ppb)",
            {"2min": 0.8, "10min": 2.4, "60min": 3.1}[window],
            float(found["ch4"]["clean_level_ch4"].values[0, WINDOWS.index(window), 0]),
            "{:.1f}",
            AT_DOCUMENTED_DAY and QUANTILE == 0.05,
            "at q 0.05 on 20240718",
        )
table_c = {"ch4": 0.59, "co2": 0.70, "noy": 0.77, "benzene_ptr": 0.40}
table_b = {"ch4": 0.68, "co2": 0.46, "noy": 0.59, "benzene_ptr": 0.40}
for v in variables:
    documented(
        "§6.8",
        f"the drive's {v}: share of readings in events, C",
        table_c[v],
        share_c[v],
        "{:.0%}",
        AT_RULE,
        "at 10 min, q 0.05 on 20240718",
    )
    documented(
        "§6.8",
        f"the drive's {v}: share of readings in events, B",
        table_b[v],
        share_b[v],
        "{:.0%}",
        AT_RULE,
        "at 10 min, q 0.05 on 20240718",
    )
at = {
    v: catalogs[v][catalogs[v]["baseline_window"] == pd.Timedelta(WINDOW).total_seconds()]
    for v in variables
}
ones = [(at[v]["n_readings"] == 1).mean() for v in variables]
twos = [(at[v]["n_readings"] <= 2).mean() for v in variables]
documented(
    "§6.8",
    "the drive: events of one reading, across the four",
    "2.6–15.5 %",
    f"{100 * min(ones):.1f}–{100 * max(ones):.1f} %",
    "{}",
    AT_RULE,
    "at 10 min on 20240718",
)
documented(
    "§6.8",
    "the drive: events of two readings or fewer",
    "6.7–30.1 %",
    f"{100 * min(twos):.1f}–{100 * max(twos):.1f} %",
    "{}",
    AT_RULE,
    "at 10 min on 20240718",
)
documented(
    "§6.8",
    "the drive's Picarro methane: median covered share under half",
    "yes",
    "yes" if at["ch4"]["covered"].median() < 0.5 else "no",
    "{}",
    AT_RULE,
    "at 10 min on 20240718",
)
documented(
    "§6.8",
    "chance events at entry 3 on 8,448 readings",
    11,
    float(found["ch4"]["chance_events_ch4"].values[w, 0, 0, 0]),
    "{:.0f}",
    AT_RULE,
    "on the 20240718 Picarro",
)
documented_noise = {
    "co2": (6.5, 30),
    "noy": (3.9, 17),
    "ch4": (2.0, 4.3),
    "benzene_ptr": (1.7, 3.7),
}
for v, (typical, busiest) in documented_noise.items():
    q5, q50, q95 = np.nanpercentile(local[v], [5, 50, 95])
    documented(
        "§2.5",
        f"the drive's {v}: diff_mad typical ÷ quietest 5 %",
        typical,
        q50 / q5,
        "{:.1f}",
        AT_RULE,
        "over 10 min on 20240718",
    )
    documented(
        "§2.5",
        f"the drive's {v}: diff_mad busiest 5 % ÷ quietest 5 %",
        busiest,
        q95 / q5,
        "{:.2g}" if busiest >= 10 else "{:.1f}",
        AT_RULE,
        "over 10 min on 20240718",
    )

# NOy's reported column as a floor under the clean spread: how often it would set the threshold.
d_noy = states["noy"]["enhancement_noy"].values
cells = stream_cells(readings["noy"], "lif")
one = find_records(cells, np.ones(d_noy.shape[0], bool), gap_ns=24 * HOUR, max_length_ns=24 * HOUR)
for k, window in enumerate(WINDOWS):
    m = float(found["noy"]["clean_level_noy"].values[0, k, 0])
    c = float(found["noy"]["clean_spread_noy"].values[0, k, 0])
    doc = {"2min": 1.00, "10min": 1.00, "60min": 0.90}.get(window)
    if doc is not None:
        documented(
            "§6.8",
            f"the drive's NOy: reported column above the clean spread at {window}",
            doc,
            float(np.mean(noy_reported > c)),
            "{:.0%}",
            AT_RULE,
            "on 20240718",
        )
    if window == "2min":
        with_floor = find_events(
            (d_noy[:, k, 0] - m) / np.maximum(c, noy_reported),
            cells,
            one,
            enter=3.0,
            exit_=1.0,
            max_internal_gap_ns=0,
        )
        without = find_events(
            (d_noy[:, k, 0] - m) / c, cells, one, enter=3.0, exit_=1.0, max_internal_gap_ns=0
        )
        documented(
            "§6.8",
            "the drive's NOy at 2 min: share in events, clean spread alone → with the floor",
            "75 → 54 %",
            f"{100 * np.mean(without.membership >= 0):.0f} → "
            f"{100 * np.mean(with_floor.membership >= 0):.0f} %",
            "{}",
            AT_RULE,
            "on 20240718",
        )

# Checks: C's events are the definition's, reading by reading, for the lead variables.
for v in ("ch4", "benzene_ptr"):
    cells = stream_cells(readings[v], variables[v])
    n = readings[v].sizes["time"]
    records = np.zeros(n, dtype=int)
    loop = reference_events(
        found[v][f"z_{v}"].values[:, w, 0],
        cells,
        reference_dropouts(cells.midpoint_ns, records),
        records,
        3.0,
        1.0,
        1,
    )
    check(
        f"§4 {v}'s events at {WINDOW} are a loop over the definition's, reading by reading",
        np.array_equal(found[v][f"event_{v}"].values[:, w, 0, 0, 0], loop),
    )

### The canister takes its events from the PTR

In [ ]:
# ---- PARAMETERS (canister) ----------------------------------------------------
CANISTER_WINDOW = "10min"
TRIGGER = "ptr.benzene_ptr"  # any dense variable, of any field

In [ ]:
canister = drive["iwas"]
fills = int(np.isfinite(canister["benzene_iwas"].values).sum())
# A declared constant of zero: the canister's enhancement is its reading (notebook 05 §6).
zero = {f"iwas.{v}": {"method": "constant", "value": 0.0} for v in ("benzene_iwas", "toluene_iwas")}
settings = sweep((CANISTER_WINDOW,), (QUANTILE,), methods=zero)
dense, trigger_variable = TRIGGER.split(".")
bases = {
    "iwas": baseline_state(
        canister, instrument="iwas", baseline=settings, variables=["benzene_iwas", "toluene_iwas"]
    ),
    dense: baseline_state(
        finite_only(drive[dense], trigger_variable),
        instrument=dense,
        baseline=settings,
        variables=[trigger_variable],
    ),
}
alone = event_state(bases["iwas"], instrument="iwas", events=EventsConfig())
below = int(alone["n_clean_readings_benzene_iwas"].values.max())
print(
    f"the canister alone: {fills} fills, {below} of them below its clean level, "
    "against the 100 a record needs"
)
documented(
    "§6.8",
    "the drive's canister: fills below its clean level",
    "7 of 32",
    f"{below} of {fills}",
    "{}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)

taken = event_states(bases, EventsConfig(triggers={"iwas": TRIGGER}))
table = event_catalog(taken, bases)
can = table[table["instrument"] == "iwas"]
trig = table[table["instrument"] == dense].set_index("event_id")
middle = canister["time"].values
print(
    f"the {dense} found {len(trig)} events at {CANISTER_WINDOW}; the canister's rows: "
    f"{can['event_id'].str.contains('benzene_iwas').sum()} for benzene, "
    f"{can['event_id'].str.contains('toluene_iwas').sum()} for toluene, "
    "each an event holding a fill"
)
in_event = taken["iwas"]["event_benzene_iwas"].values[:, 0, 0, 0, 0] >= 0
print(f"fills inside an event: {in_event.sum()} of {fills}")
theirs = trig.loc[can["trigger_event_id"]]
check(
    "§4 every canister row is its trigger's interval, "
    "holding exactly the fills whose midpoints lie in it",
    bool(
        (can["start_time"].to_numpy() == theirs["start_time"].to_numpy()).all()
        and (can["end_time"].to_numpy() == theirs["end_time"].to_numpy()).all()
    )
    and can["n_readings"].tolist()
    == [
        int(((middle >= a) & (middle < b)).sum())
        for a, b in zip(can["start_time"], can["end_time"])
    ],
    f"{len(can)} rows",
)

In [ ]:
# Figure only: the PTR's benzene over the drive, its events, and the canister's fills.
ptr = bases[dense]
t = ptr["time"].values
fig, ax = plt.subplots(figsize=(10, 4.0))
shade(ax, trig.reset_index())
clock, values = break_gaps(t, ptr[trigger_variable].values)
ax.plot(clock, values, color=MUTED, lw=0.6, label=f"{TRIGGER}, one reading a second")
value = canister["benzene_iwas"].values
ax.plot(
    middle[in_event], value[in_event], "o", color=C1, ms=6, label="canister fill inside an event"
)
ax.plot(
    middle[~in_event],
    value[~in_event],
    "x",
    color=C2,
    ms=7,
    mew=1.6,
    label="canister fill outside every event",
)
high = np.nanpercentile(ptr[trigger_variable].values, 99.5)
ax.set_ylim(-0.1, max(high, np.nanmax(value)) * 1.15)
hhmm(ax)
finish(
    ax,
    f"The {DRIVE_DAY} drive: the canister's benzene takes the PTR's events",
    f"shaded: the PTR's events at {CANISTER_WINDOW}; "
    "the canister is filled for 15 s every few minutes",
    "benzene (ppb)",
    legend=True,
    loc="upper left",
)
plt.show()

**What to read off the output.** Plume-free air's level grows with the window,
as METHODS measured, because a longer window's low quantile reaches further
down into the air's wobble; that is why the level is measured and not modelled.
Under the clean spread about three fifths of the drive's methane and two fifths
of its benzene sit inside an event: a drive through a city is mostly plume. A
handful of events are a single reading, and they stay: width cannot tell a
chance crossing from a narrow plume. The ledger rows above re-measure every
drive number METHODS quotes, including the rejected rolling noise estimate and
the NOy column that would have set the threshold had it been used as a floor.

The canister has far fewer fills below its level than a record needs, so on its
own it has no description of clean air and finds nothing. Named with a trigger,
it takes the PTR's events: each canister row is a PTR event that holds a fill,
and the figure shows which fills were taken inside the PTR's plumes.

**Try it.** `DRIVE_DAY = "20240730"` (and re-run from the loading cell): another
drive, every check still holds, and the ledger shows its numbers without
comparing them. `TRIGGER = "picarro.ch4"`: methane's events decide the
canister's, a trigger of another field (`METHODS.md` §6.8).

---
## 5. Ledger and checks

Every number `docs/METHODS.md` §6.8 and §2.5 quote about the archive,
re-measured above, and every ✔ check as it ran.

In [ ]:
ledger = pd.DataFrame(
    [(claim, *row) for claim, row in LEDGER.items()],
    columns=["claim", "METHODS", "documented", "measured", "status"],
)
with pd.option_context("display.max_rows", None, "display.max_colwidth", 70, "display.width", 200):
    print(ledger[["METHODS", "claim", "documented", "measured", "status"]].to_string(index=False))
counts = ledger["status"].value_counts()
print(
    f"\nledger: {counts.get('agrees', 0)} agree, {counts.get('DIFFERS', 0)} differ, "
    f"{counts.get('not compared', 0)} not compared, of {len(ledger)}"
)
print(f"checks: {sum(CHECKS.values())} of {len(CHECKS)} hold")
for claim, held in CHECKS.items():
    if not held:
        print(f"  ✘ {claim}")